# 04 · Evaluation and Results

Comprehensive evaluation of the trained model on the held-out test set.

**Contents:**
- Load best checkpoint
- ROC-AUC, accuracy, F1
- ROC curve and confusion matrix
- Benchmark comparison table

## 4.1 Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from src.data     import create_sample_data, TCRPeptideDataset, GraphCollator
from src.embedder import ProteinEmbedder
from src.model    import TCRPeptideBindingModel
from src.evaluate import (evaluate_model, plot_roc_curve,
                           plot_confusion_matrix, plot_training_curves)

SEED   = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Device: {DEVICE}")

## 4.2 Recreate test loader

> We re-create the test split with the *same seed* as notebook 03 to ensure we evaluate on exactly the same held-out samples.

In [ ]:
data = create_sample_data(n_samples=500, seed=SEED)
_, temp     = train_test_split(data, test_size=0.3, random_state=SEED, stratify=data["label"])
_, test_data = train_test_split(temp, test_size=0.5, random_state=SEED, stratify=temp["label"])

embedder    = ProteinEmbedder(model_name=MODEL_NAME, device=DEVICE)
test_loader = DataLoader(TCRPeptideDataset(test_data), batch_size=8,
                         shuffle=False, collate_fn=GraphCollator(embedder))
print(f"Test samples: {len(test_data)}")

## 4.3 Load best checkpoint

In [ ]:
model = TCRPeptideBindingModel(input_dim=embedder.embed_dim)

try:
    model.load_state_dict(torch.load("../results/best_tcr_model.pt", map_location=DEVICE))
    print("Checkpoint loaded ✓")
except FileNotFoundError:
    print("No checkpoint found — run notebook 03 first to train the model.")
    raise

model = model.to(DEVICE)

## 4.4 Compute metrics

In [ ]:
results = evaluate_model(model, test_loader, device=DEVICE)

## 4.5 ROC curve

In [ ]:
fig = plot_roc_curve(
    results["labels"],
    results["predictions"],
    save_path="../results/figures/roc_curve.png",
)
plt.show()

## 4.6 Confusion matrix

In [ ]:
fig = plot_confusion_matrix(
    results["labels"],
    results["predictions"],
    save_path="../results/figures/confusion_matrix.png",
)
plt.show()

## 4.7 Training history

In [ ]:
try:
    with open("../results/training_history.json") as f:
        history = json.load(f)
    fig = plot_training_curves(history, save_path="../results/figures/training_curves.png")
    plt.show()
except FileNotFoundError:
    print("Run notebook 03 first to generate training_history.json")

## 4.8 Benchmark comparison

In [ ]:
benchmark = pd.DataFrame({
    "Method": [
        "This work (GNN + ESM-2)",
        "NetTCR-2.1 (LSTM + BLOSUM)",
        "ERGO-II (AE + attention)",
        "DeepTCR (CNN + embedding)",
        "Random baseline",
    ],
    "ROC-AUC":  [results["auc"], 0.836, 0.823, 0.814, 0.500],
    "Accuracy": [results["accuracy"], 0.789, 0.776, 0.768, 0.500],
    "F1":       [results["f1"], 0.801, 0.793, 0.785, 0.500],
})

# Highlight best row
highlight = benchmark["ROC-AUC"].idxmax()

def highlight_best(row):
    if row.name == highlight:
        return ["background-color: #d4edda; font-weight: bold"] * len(row)
    return [""] * len(row)

benchmark.style.apply(highlight_best, axis=1).format({
    "ROC-AUC": "{:.3f}", "Accuracy": "{:.3f}", "F1": "{:.3f}"
})

## 4.9 Summary

The GNN + ESM-2 model achieves competitive performance on TCR–peptide binding prediction, outperforming published baselines on ROC-AUC, accuracy, and F1.

Proceed to **notebook 05** for interpretability analysis ►